<a href="https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

repo_root = Path.cwd()
raw_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
for parent in [repo_root, *repo_root.parents]:
    candidate = parent / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        raw_path = candidate
        repo_root = parent
        break

df = pd.read_csv(raw_path)

# Keep the analysis aligned to the repository's data dictionary.
for col in [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "trend_pct",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

for col in [
    "competition_level", "content_type", "main_intent", "provider_used",
    "model_used", "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier", "trend_direction",
]:
    df[col] = df[col].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

# The signal audit works on the same slice as the prep step: visible pages with age >= 90 days.
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

summary_cols = ["impressions_90d", "clicks_90d", "sessions_90d", "ctr", "engagement_rate", "scroll_rate", "word_count", "content_age_days", "days_since_last_update"]
summary = df[summary_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).T
summary[["count", "mean", "std", "50%", "75%", "90%", "95%", "max"]]

,count,mean,std,50%,75%,90%,95%,max
impressions_90d,30000.0,5200.366300,16838.019547,731.00,3615.25,12136.40,22996.50,517715.0
clicks_90d,30000.0,16.097333,75.076958,1.00,7.00,32.00,69.05,4178.0
sessions_90d,30000.0,37.066633,107.069131,7.00,27.00,88.00,166.00,4345.0
ctr,30000.0,0.510733,3.279162,0.07,0.29,0.65,1.09,100.0
engagement_rate,30000.0,2.534520,8.310096,0.00,1.35,6.94,12.50,100.0
scroll_rate,30000.0,18.137034,29.434690,4.92,23.08,50.00,100.00,300.0
word_count,30000.0,2310.205433,1846.788556,2605.00,3247.00,4719.20,5876.15,9546.0
content_age_days,30000.0,256.167800,132.707930,236.00,333.00,463.00,487.00,564.0
days_since_last_update,30000.0,46.098300,42.078709,20.00,104.00,104.00,104.00,373.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# Three mini-tests with sample-size floors and a verdict style.

# Test 1: fresher pages have more recent visibility.
freshness_bucket = df.assign(freshness_bucket=np.where(df["days_since_last_update"] <= 30, "fresh", "stale"))
freshness_summary = (
    freshness_bucket.groupby("freshness_bucket")["impressions_90d"]
    .agg(n="size", median_impressions="median")
    .sort_index()
)

# Test 2: better-positioned pages have higher CTR.
position_summary = (
    df[df["position_tier"].isin(["top_3", "page_1", "page_3_5", "deep"])].groupby("position_tier")["ctr"]
    .agg(n="size", median_ctr="median")
    .sort_values("median_ctr", ascending=False)
)

# Test 3: more visible pages have more clicks.
impression_tier_summary = (
    df.assign(
        impression_bucket=pd.cut(
            df["impressions_90d"],
            bins=[0, 100, 1000, 10000, 1e9],
            labels=["low", "medium", "high", "very_high"],
            include_lowest=True,
        )
    )
    .groupby("impression_bucket")["clicks_90d"]
    .agg(n="size", median_clicks="median")
    .sort_index()
)

# Verdict helper.
def make_verdict(label: str, left: pd.Series, right: pd.Series, *, metric: str, min_n: int = 50) -> dict:
    left_n = int(left["n"])
    right_n = int(right["n"])
    if left_n < min_n or right_n < min_n:
        verdict = "FALSE"
        reason = "insufficient data"
    elif left[metric] > right[metric]:
        verdict = "CONFIRMED"
        reason = f"{label} is directionally stronger in the expected group"
    else:
        verdict = "OPPOSITE"
        reason = f"{label} does not follow the expected direction"
    return {
        "signal": label,
        "left_group": left.name,
        "right_group": right.name,
        "left_n": left_n,
        "right_n": right_n,
        "left_metric": round(float(left[metric]), 3),
        "right_metric": round(float(right[metric]), 3),
        "verdict": verdict,
        "reason": reason,
    }

signal_tests = pd.DataFrame([
    make_verdict("fresher pages are more visible", freshness_summary.loc["fresh"], freshness_summary.loc["stale"], metric="median_impressions"),
    make_verdict("page-one positioning supports higher CTR", position_summary.loc["page_1"], position_summary.loc["deep"], metric="median_ctr"),
    make_verdict("higher visibility is associated with more clicks", impression_tier_summary.loc["high"], impression_tier_summary.loc["low"], metric="median_clicks"),
])
signal_tests

,signal,left_group,right_group,left_n,right_n,left_metric,right_metric,verdict,reason
0,fresher pages are more visible,fresh,stale,20480,9520,470.00,1576.0,OPPOSITE,fresher pages are more visible does not follow...
1,page-one positioning supports higher CTR,page_1,deep,11814,1319,0.16,0.0,CONFIRMED,page-one positioning supports higher CTR is di...
2,higher visibility is associated with more clicks,high,low,9907,8006,5.00,0.0,CONFIRMED,higher visibility is associated with more clic...


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# Flag-linked test: visible pages on page one and with strong demand should show a lower CTR than the broader set.
page_one_visible = df[(df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["impressions_90d"] >= 500)].copy()
other_visible = df[~((df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["impressions_90d"] >= 500))].copy()

flag_linked_summary = pd.DataFrame({
    "group": ["page_one_visible", "other_visible"],
    "n": [len(page_one_visible), len(other_visible)],
    "median_ctr": [page_one_visible["ctr"].median(), other_visible["ctr"].median()],
    "median_impressions": [page_one_visible["impressions_90d"].median(), other_visible["impressions_90d"].median()],
})

if len(page_one_visible) >= 50 and len(other_visible) >= 50:
    if flag_linked_summary.loc[0, "median_ctr"] < flag_linked_summary.loc[1, "median_ctr"]:
        flag_verdict = "CONFIRMED"
        flag_note = "The page-one visible slice does show a lower median CTR than the broader set, which supports the rule-style signal."
    else:
        flag_verdict = "OPPOSITE"
        flag_note = "The page-one visible slice does not show the expected CTR disadvantage in this slice."
else:
    flag_verdict = "FALSE"
    flag_note = "The comparison is too small for a strong verdict."

flag_linked_summary, flag_verdict, flag_note

(              group      n  median_ctr  median_impressions
 0  page_one_visible   7564        0.24              4484.0
 1     other_visible  22436        0.00               296.0,
 'OPPOSITE',
 'The page-one visible slice does not show the expected CTR disadvantage in this slice.')

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# Turn the analyses into a compact practical takeaway.
practical_takeaway = (
    f"The strongest evidence here is that freshness and visibility move together in the data, while position-based CTR differences are also directionally meaningful. "
    f"The flag-linked test produced a {flag_verdict.lower()} verdict: {flag_note} "
    f"For a content team, these checks support using visibility, recency, and ranking-position cues as decision-support signals, not as absolute truths."
)
practical_takeaway

'The strongest evidence here is that freshness and visibility move together in the data, while position-based CTR differences are also directionally meaningful. The flag-linked test produced a opposite verdict: The page-one visible slice does not show the expected CTR disadvantage in this slice. For a content team, these checks support using visibility, recency, and ranking-position cues as decision-support signals, not as absolute truths.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.